In [1]:
# Loading the data
import json
import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 150)

with open("../data/cuad/train_separate_questions.json") as f:
    train = json.load(f)
with open("../data/cuad/test.json") as f:
    test = json.load(f)

print("train contracts:", len(train["data"]))
print("test contracts:", len(test["data"]))

train contracts: 408
test contracts: 102


In [17]:
# 1. Presence table: one row per (contract, category)
rows = []
for split, data in [("train", train), ("test", test)]:
    for doc in data["data"]:
        # Count how many spans each category has in this contract.
        spans = {}
        for qa in doc["paragraphs"][0]["qas"]:
            category = qa["question"].split('"')[1]
            spans[category] = spans.get(category, 0) + len(qa["answers"])

        for category, n in spans.items():
            rows.append({"split": split,
                         "contract": doc["title"],
                         "category": category,
                         "n_spans": n,
                         "present": n > 0})

presence = pd.DataFrame(rows)

print("rows:", len(presence), "(Total 510 x 41 =", 510 * 41, ")")
presence.head(42)

rows: 20910 (Total 510 x 41 = 20910 )


,split,contract,category,n_spans,present
0,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,1,True
1,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,5,True
2,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,1,True
3,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,2,True
4,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,1,True
5,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Renewal Term,1,True
6,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Notice Period To Terminate Renewal,0,False
7,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Governing Law,1,True
8,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Most Favored Nation,0,False
9,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Non-Compete,0,False


In [8]:
# 2. Positive rate per category (train only)
train_only = presence[presence["split"] == "train"]

rate = train_only.groupby("category").agg(
    contracts_with=("present", "sum"),
    total_contracts=("present", "count"),
)
rate["pct"] = (100 * rate["contracts_with"] / rate["total_contracts"]).round(1)
rate = rate.sort_values("pct", ascending=False)

print(rate.to_string())

                                    contracts_with  total_contracts    pct
category                                                                  
Document Name                                  408              408  100.0
Parties                                        407              408   99.8
Agreement Date                                 377              408   92.4
Governing Law                                  354              408   86.8
Expiration Date                                335              408   82.1
Effective Date                                 320              408   78.4
Anti-Assignment                                302              408   74.0
Cap On Liability                               231              408   56.6
License Grant                                  205              408   50.2
Audit Rights                                   176              408   43.1
Termination For Convenience                    154              408   37.7
Post-Termination Services

In [10]:
# 3. How many categories are rare?
print("categories under 10%:", (rate["pct"] < 10).sum(), "of 41")
print("categories under 25%:", (rate["pct"] < 25).sum(), "of 41")
print("mean positive rate:", round(rate["pct"].mean(), 1), "%")
print()
print("Rarest 10 - these have too few examples for stable metrics:")
print(rate.tail(10).to_string())

categories under 10%: 9 of 41
categories under 25%: 23 of 41
mean positive rate: 32.6 %

Rarest 10 - these have too few examples for stable metrics:
                                   contracts_with  total_contracts   pct
category                                                                
Liquidated Damages                             47              408  11.5
Joint Ip Ownership                             39              408   9.6
Non-Disparagement                              31              408   7.6
No-Solicit Of Customers                        27              408   6.6
Third Party Beneficiary                        26              408   6.4
Most Favored Nation                            25              408   6.1
Affiliate License-Licensor                     17              408   4.2
Price Restrictions                             15              408   3.7
Unlimited/All-You-Can-Eat-License              14              408   3.4
Source Code Escrow                             1

In [12]:
# 4. Contracts with a category vs total spans for it
compare = train_only.groupby("category").agg(
    contracts=("present", "sum"),
    spans=("n_spans", "sum"),
)
compare["pct_of_contracts"] = (100 * compare["contracts"] / 408).round(1)
compare["pct_of_spans"] = (100 * compare["spans"] / compare["spans"].sum()).round(1)
compare["spans_per_contract"] = (
    compare["spans"] / compare["contracts"].replace(0, pd.NA)
).round(2)
compare = compare.sort_values("pct_of_spans", ascending=False)

print("total spans:", compare["spans"].sum())
print(compare.to_string())


total spans: 11180
                                    contracts  spans  pct_of_contracts  pct_of_spans  spans_per_contract
category                                                                                                
Parties                                   407   2011              99.8          18.0                4.94
License Grant                             205    642              50.2           5.7                3.13
Cap On Liability                          231    554              56.6           5.0                2.40
Audit Rights                              176    538              43.1           4.8                3.06
Anti-Assignment                           302    517              74.0           4.6                1.71
Insurance                                 134    443              32.8           4.0                3.31
Document Name                             408    419             100.0           3.7                1.03
Agreement Date                      

In [14]:
# Span count per category, only in contracts where it appears
found = train_only[train_only["present"]]

spread = found.groupby("category")["n_spans"].agg(
    contracts="count", mean="mean", median="median", min="min", max="max"
).round(2).sort_values("max", ascending=False)

print(spread.to_string())


                                    contracts  mean  median  min  max
category                                                             
Parties                                   407  4.94     4.0    2   55
Post-Termination Services                 153  2.41     1.0    1   23
Rofr/Rofo/Rofn                             68  4.40     2.0    1   22
Audit Rights                              176  3.06     2.0    1   19
Cap On Liability                          231  2.40     2.0    1   16
Insurance                                 134  3.31     2.0    1   16
License Grant                             205  3.13     2.0    1   16
Ip Ownership Assignment                   101  2.54     2.0    1   15
Minimum Commitment                        133  2.53     2.0    1   14
Non-Transferable License                  116  2.20     2.0    1   12
Non-Compete                                96  2.08     2.0    1   12
Exclusivity                               147  2.26     2.0    1   12
Revenue/Profit Shari

`spans_per_contract` is why this is not one-label-per-contract. `Parties` averages
about 5 spans wherever it appears, and `Source Code Escrow` appears in only 12
contracts but carries 61 spans between them. A rare category is not necessarily a
category with few spans.

In [15]:
# 5. Did the official split skew the label distribution?
by_split = presence.groupby(["category", "split"])["present"].mean().unstack() * 100
by_split = by_split.round(1)
by_split["gap"] = (by_split["test"] - by_split["train"]).abs().round(1)

print("Largest train/test gaps (percentage points):")
print(by_split.sort_values("gap", ascending=False).head(10).to_string())
print()
print("mean absolute gap:", round(by_split["gap"].mean(), 1), "pp")

# No contract should appear in both splits.
overlap = set(presence[presence["split"] == "train"]["contract"]) & \
          set(presence[presence["split"] == "test"]["contract"])
print("contracts in both splits:", len(overlap))

Largest train/test gaps (percentage points):
split                               test  train   gap
category                                             
Cap On Liability                    43.1   56.6  13.5
Renewal Term                        25.5   36.8  11.3
Uncapped Liability                  12.7   24.0  11.3
Effective Date                      68.6   78.4   9.8
Termination For Convenience         28.4   37.7   9.3
Post-Termination Services           28.4   37.5   9.1
Notice Period To Terminate Renewal  15.7   23.3   7.6
Non-Transferable License            21.6   28.4   6.8
Warranty Duration                    9.8   15.9   6.1
Audit Rights                        37.3   43.1   5.8

mean absolute gap: 3.6 pp
contracts in both splits: 0


Mean gap is 3.6 pp and no contract appears in both splits, so the official split is
safe to use as-is. The larger gaps (Cap On Liability, 13.5 pp) are small-sample noise
on 102 test contracts, not a distribution shift.

In [16]:
# 6. Naive baseline: predict "absent" for every category
y = train_only["present"]

accuracy = (y == False).mean()
recall = 0.0                      # finds nothing, by definition

print("Baseline: predict 'absent' for every category")
print(f"  accuracy: {accuracy * 100:.1f}%")
print(f"  recall:   {recall * 100:.1f}%")
print()
print(f"A model can look {accuracy * 100:.0f}% accurate and find zero clauses.")

Baseline: predict 'absent' for every category
  accuracy: 67.4%
  recall:   0.0%

A model can look 67% accurate and find zero clauses.


## 7. Metric decision

The numbers above settle how we evaluate, so the October modelling work does not have
to revisit it.

**Findings**

- Mean positive rate is 32%. Most (contract, category) pairs are negatives.
- 9 of 41 categories appear in under 10% of contracts; 23 of 41 in under 25%.
- The rarest category, Source Code Escrow, appears in 12 of 408 contracts.
- Predicting "absent" everywhere scores 67.4% accuracy with 0% recall.

**Decision**

- **Do not report accuracy as a headline.** 67.4% is achievable by a model that finds
  nothing, so accuracy measures the imbalance rather than the model.
- **Report per-category precision, recall and F1, always with support** (the number of
  positive contracts). F1 on a 12-contract category is not comparable to F1 on a
  400-contract one, and support is what makes that visible.
- **Use macro-F1 as the single headline number.** It weights all 41 categories equally,
  so the rare ones cannot be ignored.
- **Treat the 9 categories under 10% separately** in error analysis. Expect unstable
  metrics there and say so rather than averaging it away.
- **Baseline to beat:** "predict absent" at 67.4% accuracy / 0% recall. Quote both
  numbers together in the final report.

Handling the imbalance during training (class weights, resampling, threshold tuning)
is October work.